# Rainfall Analysis and Planting Advisory: Five Kenyan Counties

**Author:** Buhle Pearl Nkosi · 1MillionDevs Cohort Data Analytics

**Business question:** A farmer-support NGO sends SMS advice on planting windows in five counties. Which months should each county plant in, and how reliable is the data behind that advice?

**Lifecycle:** Ask → Prepare → Process (clean Machakos) → Analyse (NumPy) → Share (CSV + advisory) → Act.

## 1. Prepare: load the data

Rows are counties and columns are months (January to December), in mm of rainfall. The Machakos row arrived as hand-typed text and needs cleaning first.

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 2. Process: clean the Machakos row

The function tries to convert each value with `int()`. A bad value such as `"N/A"` or an empty string raises a `ValueError`, which we catch specifically (not a bare `except`). Bad months are replaced with the rounded average of the valid Machakos values, and the imputed months are printed. `int(" 20 ")` works because `int()` ignores surrounding spaces.

In [2]:
def clean_rainfall(raw_values, month_names):
    """Convert raw text readings to ints; impute invalid ones with the rounded mean.

    Returns the cleaned list and the list of imputed month names.
    """
    converted = []
    bad_positions = []
    for i, value in enumerate(raw_values):
        try:
            converted.append(int(value))
        except ValueError:
            converted.append(None)
            bad_positions.append(i)

    valid = [v for v in converted if v is not None]
    fill_value = round(sum(valid) / len(valid))

    cleaned = [fill_value if v is None else v for v in converted]
    imputed_months = [month_names[i] for i in bad_positions]
    return cleaned, imputed_months, fill_value


machakos_clean, imputed_months, fill_value = clean_rainfall(machakos_raw, months)

print("Imputed months:", imputed_months)
print("Imputed value (mean of valid readings, rounded):", fill_value, "mm")
print("Cleaned Machakos row:", machakos_clean)

Imputed months: ['Apr', 'Sep']
Imputed value (mean of valid readings, rounded): 68 mm
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Combine into one 5 × 12 NumPy array

In [3]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Shape:", rainfall.shape)
print("dtype:", rainfall.dtype)
print(rainfall)

Shape: (5, 12)
dtype: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## 4. Analyse with NumPy

### 4.1 Annual totals and monthly averages
`axis=1` collapses the months (one total per county); `axis=0` collapses the counties (one average per month).

In [4]:
annual_totals = rainfall.sum(axis=1)
monthly_avg = rainfall.mean(axis=0)

for county, total in zip(counties, annual_totals):
    print(f"{county:<9} annual total: {total:,} mm")

print()
for month, avg in zip(months, monthly_avg):
    print(f"{month}: {avg:.1f} mm")

Nairobi   annual total: 1,045 mm
Nakuru    annual total: 916 mm
Kisumu    annual total: 1,371 mm
Mombasa   annual total: 1,075 mm
Machakos  annual total: 811 mm

Jan: 47.0 mm
Feb: 47.6 mm
Mar: 91.0 mm
Apr: 164.4 mm
May: 171.2 mm
Jun: 67.6 mm
Jul: 56.6 mm
Aug: 58.2 mm
Sep: 64.2 mm
Oct: 67.8 mm
Nov: 127.4 mm
Dec: 80.6 mm


### 4.2 Wettest month per county
`np.argmax(..., axis=1)` returns the position of the largest value in each row, and that position is used to look up the month name.

In [5]:
wettest_idx = np.argmax(rainfall, axis=1)
wettest_month = np.array(months)[wettest_idx]

for county, month, mm in zip(counties, wettest_month, rainfall.max(axis=1)):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


### 4.3 Dry months (below 50 mm)
A boolean mask marks every cell under 50 mm, and summing along `axis=1` counts the `True` values per county.

In [6]:
dry_mask = rainfall < 50
dry_months = dry_mask.sum(axis=1)

for county, n in zip(counties, dry_months):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


### 4.4 Planting window: best two consecutive months
`rainfall[:, :-1] + rainfall[:, 1:]` adds each month to the next one for all counties at once. The largest pair sum per row is the recommended window.

In [7]:
pair_sums = rainfall[:, :-1] + rainfall[:, 1:]      # shape (5, 11)
best_start = np.argmax(pair_sums, axis=1)
best_total = pair_sums.max(axis=1)

months_arr = np.array(months)
planting_window = np.char.add(np.char.add(months_arr[best_start], "-"),
                              months_arr[best_start + 1])

for county, window, mm in zip(counties, planting_window, best_total):
    print(f"{county:<9} planting window: {window} ({mm} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## 5. Share: write `rainfall_summary.csv`
The file is written with Python's `csv` module: one header row and exactly five county rows.

In [8]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for county, total, wet, dry, window in zip(
            counties, annual_totals, wettest_month, dry_months, planting_window):
        writer.writerow([county, int(total), str(wet), int(dry), str(window)])

with open("rainfall_summary.csv") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,Apr,4,Apr-May
Nakuru,916,Apr,3,Apr-May
Kisumu,1371,Apr,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,Nov,6,Nov-Dec



## Planting Advisory

Nairobi, Nakuru, Kisumu and Mombasa all have their best two-month planting window in April to May, so farmers there should prepare land and have seed ready by late March. Machakos is different: its rains peak in November to December, so SMS advice for Machakos should point farmers to that window rather than the April rains. Kisumu is the most reliable county, with no dry months and 1,371 mm a year, while Machakos is the riskiest, with six dry months and the lowest annual total of 811 mm. Nakuru has a weaker and more spread-out rainy season (265 mm in its best two months), so farmers there should favour drought-tolerant varieties and plan for some dry months. Because Machakos rainfall is the least certain, the NGO should encourage short-season crops there and send a reminder if the November rains start late. Please read these windows alongside the data-quality caveat below.

## Data Quality Notes

- **Machakos data was hand-typed and partly invalid.** Two of twelve readings (April and September) were `"N/A"` or blank. I replaced both with the rounded mean of the ten valid readings (68 mm), so they are estimates, not measurements.
- **The imputation affects the advice.** The estimated 68 mm in April and September pulls the Machakos total up slightly and could hide a genuinely dry April. The Machakos planting window (Nov-Dec, 278 mm) does not depend on the imputed months, but the annual total and dry-month count are less certain than for the other four counties.
- **A text value with spaces (`" 20 "`) was valid** and converted to 20 without being treated as an error.
- **Each county is a single weather station for one year.** Rainfall varies from year to year and within a county, so these windows are a starting point and should be checked against several seasons and local forecasts before being sent as firm SMS advice.